In [16]:
import json
import os
import shutil
from pathlib import Path

import yaml
from rdflib import Dataset, Graph
from pyshacl import validate

SRCDIR = Path(".")
BUILTDIR = SRCDIR / ".." / "built"
DATADIR = Path("data")

print(SRCDIR)

def yaml2json(y):
    print(BUILTDIR / y.parent)
    os.makedirs(BUILTDIR / y.parent, exist_ok=True)
    with y.open("rb") as f:
        data = yaml.safe_load(f)
    with open(BUILTDIR / y.with_suffix(".jsonld"), "w") as g:
        json.dump(data, g)

shutil.rmtree(BUILTDIR, ignore_errors=True)

yaml2json(SRCDIR / "remote" / "base-voltage.shacl.yamlld")
yaml2json(SRCDIR / "remote" / "nbnl-shacl.context.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.rdfs.yamlld")
yaml2json(SRCDIR / "nbnl-equipment.shacl.yamlld")
shutil.copy(SRCDIR / "remote" / "cim17.rdfs.ttl", BUILTDIR / "remote")

data_dataset = Dataset()
data_dataset.parse(
    DATADIR / "AS_EQ_OP_Westland_FGU" / "equipment_four_stations_EQ.jsonld",
    format="json-ld",
)

shapes_graph = Graph()
shapes_graph.parse(BUILTDIR / "nbnl-equipment.shacl.jsonld", format="json-ld")

cim_ontology_graph = Graph()
cim_ontology_graph.parse(
    BUILTDIR / "remote" / "cim17.rdfs.ttl", format="turtle"
)  # TODO: Should be the profile vocab which imports this.

conforms, results_graph, results_text = validate(
    data_graph=data_dataset,
    shacl_graph=shapes_graph,
    ont_graph=cim_ontology_graph,  # TODO: This is very specific to this validator. Can we do this differently?
    meta_shacl=True,
    inference="rdfs",
    do_owl_imports=True,
)

print(results_text)

.
../built/remote
../built/remote
../built
../built
Validation Report
Conforms: False
Results (60):
Constraint Violation in ClassConstraintComponent (http://www.w3.org/ns/shacl#ClassConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class nl:UpdateInfoKind ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path nl:UpdateInfo.type ]
	Focus Node: <urn:uuid:18e34612-8393-5173-a514-3de0c941dfcd>
	Value Node: nl:UpdateInfoKind.ATC
	Result Path: nl:UpdateInfo.type
	Message: Value does not have class nl:UpdateInfoKind
Constraint Violation in MinCountConstraintComponent (http://www.w3.org/ns/shacl#MinCountConstraintComponent):
	Severity: sh:Violation
	Source Shape: [ sh:class cim:ConnectivityNodeContainer ; sh:maxCount Literal("1", datatype=xsd:integer) ; sh:minCount Literal("1", datatype=xsd:integer) ; sh:path cim:ConnectivityNode.ConnectivityNodeContainer ]
	Focus Node: <urn:uuid:05c55d27-4689-4729-98e5-f3b323d703ff>
	Result 